In [ ]:
# 1. Устанавливаем библиотеки для криптографии (нужны один раз)
!pip install pyjwt cryptography -q

import json
import time
import jwt
import requests

# 2. Читаем скачанный ключ
with open("authorized_key.json", "r") as f:
    key_data = json.load(f)

private_key = key_data["private_key"]
service_account_id = key_data["service_account_id"]

# 3. Формируем JWT-заголовок и тело (стандарт OAuth 2.0)
now = int(time.time())
payload = {
    "iss": service_account_id,
    "aud": "https://iam.api.cloud.yandex.net/iam/v1/tokens",
    "iat": now,
    "exp": now + 3600  # Токен для обмена живет 1 час
}
headers = {"alg": "PS256", "typ": "JWT", "kid": key_data["id"]}

# 4. Подписываем JWT закрытым ключом
signed_jwt = jwt.encode(payload, private_key, algorithm="PS256", headers=headers)

# 5. Обмениваем JWT на IAM-токен
response = requests.post(
    "https://iam.api.cloud.yandex.net/iam/v1/tokens",
    json={"jwt": signed_jwt}
)

if response.status_code == 200:
    iam_token = response.json()["iamToken"]
    print(f"✅ IAM-токен получен!")
    print(f"🔑 Скопируйте эту строку в Colab Secrets под именем YANDEX_IAM_TOKEN:")
    print(f"{iam_token}")
else:
    print(f" Ошибка обмена: {response.status_code} | {response.text}")

In [ ]:
# Импорты: os (переменные окружения), time (тайминг), requests (HTTP-клиент),
# json (парсинг ответов), userdata (безопасное хранение ключей в Colab)
import os
import time
import requests
import json
from google.colab import userdata

# 1. Безопасно извлекаем токены из Colab Secrets (не коммитим в Git!)
YANDEX_IAM_TOKEN = userdata.get('YANDEX_IAM_TOKEN')
FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')

# 2. Эндпоинт и заголовки по спецификации Yandex Cloud
url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {YANDEX_IAM_TOKEN}",
    "x-folder-id": FOLDER_ID
}

# 3. Тело запроса: ограничиваем maxTokens для контроля стоимости
payload = {
    "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
    "completionOptions": {
        "stream": False,
        "temperature": 0.3,  # Низкая температура = детерминированные ответы (важно для бенчмарков)
        "maxTokens": 100     # Жесткий лимит генерации
    },
    "messages": [{"role": "user", "text": "Кратко опиши риски просроченной дебиторской задолженности в финтехе."}]
}

# 4. Замер времени и синхронный запрос
start_time = time.perf_counter()
response = requests.post(url, headers=headers, json=payload)
end_time = time.perf_counter()

# 5. Парсинг ответа и вывод метрик
if response.status_code == 200:
    data = response.json()
    result_text = data.get("result", {}).get("alternatives", [{}])[0].get("message", {}).get("text", "Нет ответа")
    latency_ms = (end_time - start_time) * 1000

    print(f"✅ Успех! Задержка: {latency_ms:.2f} мс")
    print(f"📝 Ответ: {result_text[:150]}...")
    print(f"💰 Оценка стоимости: ~{(len(result_text.split()) / 1000) * 0.002:.4f} USD (прогноз по выходным токенам)")
else:
    print(f"❌ Ошибка {response.status_code}: {response.text}")

✅ Успех! Задержка: 2304.74 мс
📝 Ответ: Риски просроченной дебиторской задолженности в финтехе включают:

1. **Финансовые потери**: несвоевременное погашение долгов может привести к потере о...
💰 Оценка стоимости: ~0.0001 USD (прогноз по выходным токенам)


In [ ]:
# Устанавливаем библиотеку
!pip install huggingface_hub -q

from huggingface_hub import InferenceClient
from google.colab import userdata
import time

# Загружаем НОВЫЙ токен
HF_TOKEN = userdata.get('HF_TOKEN')

# Используем проверенную модель
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"🔍 Проверяем доступ к модели {MODEL_NAME}...")

# Проверяем, существует ли модель
try:
    from huggingface_hub import model_info
    info = model_info(MODEL_NAME, token=HF_TOKEN)
    print(f"✅ Модель найдена: {info.id}")
    print(f"📊 Загрузка: {info.downloads:,} раз")
except Exception as e:
    print(f"❌ Модель недоступна: {e}")

# Инициализируем клиент
client = InferenceClient(
    model=MODEL_NAME,
    token=HF_TOKEN,
    timeout=120
)

prompt = "Кратко опиши риски просроченной дебиторской задолженности в финтехе."

print(f"\n🔄 Отправляем запрос...")
start_time = time.perf_counter()

try:
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        temperature=0.3
    )
    end_time = time.perf_counter()
    latency_ms = (end_time - start_time) * 1000

    answer = response.choices[0].message.content

    print(f"✅ Успех! Задержка: {latency_ms:.2f} мс")
    print(f"📝 Ответ: {answer[:200]}...")

except Exception as e:
    print(f"❌ Ошибка: {type(e).__name__}")
    print(f"💡 Детали: {str(e)}")
    print("\n💡 Попробуйте:")
    print("   1. Подождать 30 сек (cold start)")
    print("   2. Или переключиться на Yandex Cloud Qwen")

🔍 Проверяем доступ к модели Qwen/Qwen2.5-7B-Instruct...
✅ Модель найдена: Qwen/Qwen2.5-7B-Instruct
📊 Загрузка: 12,160,192 раз

🔄 Отправляем запрос...
✅ Успех! Задержка: 1081.87 мс
📝 Ответ: Просроченная дебиторская задолженность в финтех-компаниях может представлять ряд рисков:

1. **Финансовые потери**: Просроченная задолженность приводит к уменьшению доступных средств для бизнеса, что ...


In [ ]:
# Импорты для визуализации
import pandas as pd
from datetime import datetime

# 1. Собираем результаты (заполните реальными значениями из ваших тестов)
results = pd.DataFrame([
    {
        "Model": "YandexGPT (latest)",
        "Provider": "Yandex Cloud",
        "Latency_ms": 2304.74,  # ← Ваше значение из первого теста
        "Cost_per_1k_tokens_USD": 0.002,  # Тариф YC: https://yandex.cloud/ru/pricing/foundation-models
        "Max_Tokens": 100,
        "Temperature": 0.3,
        "Response_Preview": "Риски просроченной дебиторской задолженности...",
        "Test_Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
    },
    {
        "Model": "Qwen2.5-7B-Instruct",
        "Provider": "Hugging Face (Free Tier)",
        "Latency_ms": 1081.87,  # ← Ваше значение из теста с Qwen
        "Cost_per_1k_tokens_USD": 0.00,  # Бесплатно, но с лимитами
        "Max_Tokens": 100,
        "Temperature": 0.3,
        "Response_Preview": "Просроченная дебиторская задолженность...",
        "Test_Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
])

# 2. Добавляем вычисляемые метрики
results["Tokens_Estimated"] = results["Response_Preview"].apply(lambda x: len(x.split()) * 1.3)  # грубая оценка
results["Est_Cost_USD"] = (results["Tokens_Estimated"] / 1000) * results["Cost_per_1k_tokens_USD"]

# 3. Красивый вывод в Markdown (для README)
print("## 📊 LLM Benchmark Results — MVP\n")
print(results[["Model", "Provider", "Latency_ms", "Est_Cost_USD", "Response_Preview"]].to_markdown(index=False))

# 4. Сохраняем таблицу для истории (опционально)
results.to_csv("benchmark_results_mvp.csv", index=False)
print(f"\n💾 Таблица сохранена в `benchmark_results_mvp.csv`")

## 📊 LLM Benchmark Results — MVP

| Model               | Provider                 |   Latency_ms |   Est_Cost_USD | Response_Preview                                |
|:--------------------|:-------------------------|-------------:|---------------:|:------------------------------------------------|
| YandexGPT (latest)  | Yandex Cloud             |      2304.74 |       1.04e-05 | Риски просроченной дебиторской задолженности... |
| Qwen2.5-7B-Instruct | Hugging Face (Free Tier) |      1081.87 |       0        | Просроченная дебиторская задолженность...       |

💾 Таблица сохранена в `benchmark_results_mvp.csv`
